In [29]:
!pip install langchain langchain-community chromadb sentence-transformers groq langchain-groq langgraph

In [30]:
import langchain
import chromadb
from sentence_transformers import SentenceTransformer
import groq
import langgraph

print("langchain:", langchain.__version__)
print("chromadb:", chromadb.__version__)
print("groq:", groq.__version__)
print("All imports successful ✅")

langchain: 1.2.12
chromadb: 1.5.5
groq: 0.37.1
All imports successful ✅


# **Knowledge Base Documents**

In [31]:
fraud_patterns = [
    # Velocity patterns
    "High velocity transactions — more than 5 transactions in one hour from the same card indicates card testing. Fraudsters make small rapid transactions to verify a stolen card works before making a large purchase.",

    # Geographic patterns
    "Transactions originating from high-risk countries such as Nigeria, Romania, and Ukraine while the cardholder is based in USA or UK are strong fraud indicators. Card data is frequently sold on dark web markets and used overseas.",

    # Time patterns
    "Transactions occurring between midnight and 4am are statistically 3x more likely to be fraudulent. Fraudsters operate during off-hours when victims are asleep and less likely to notice unauthorized charges.",

    # Amount patterns
    "Unusually high transaction amounts above $600 on cards with average spend below $300 indicate fraud. Criminals maximize stolen card value before the card is blocked.",

    # Device patterns
    "Device mismatch — transaction made from a device not previously associated with the account — is a strong account takeover indicator. 80% of account takeover fraud involves a new unrecognized device.",

    # Distance patterns
    "Transaction location more than 500km from the cardholder's home address is a strong fraud signal, especially combined with other risk factors. Physical distance suggests card cloning or data theft.",

    # Combined patterns
    "The combination of high amount, foreign country, late night hour, and device mismatch creates a compound fraud signal. When 3 or more risk factors appear together, fraud probability exceeds 95%.",
]

regulations = [
    "PCI-DSS Requirement 10: All transactions must be logged with full audit trail including timestamp, amount, merchant, and device information for fraud investigation purposes.",

    "PCI-DSS Requirement 6: Transactions exceeding $500 from high-risk geographic regions must trigger additional verification before approval.",

    "Basel III Framework: Financial institutions must maintain real-time fraud detection systems capable of flagging suspicious transactions within 200 milliseconds to minimize financial exposure.",

    "FFIEC Guidelines: Account takeover fraud requires immediate card blocking and customer notification when device mismatch is detected alongside unusual transaction patterns.",

    "Regulation E (Electronic Fund Transfer Act): Banks must investigate reported fraudulent transactions within 10 business days and provisionally credit the customer within 5 days.",
]

# Combine everything into one list
all_documents = fraud_patterns + regulations

print(f"Fraud patterns loaded: {len(fraud_patterns)}")
print(f"Regulations loaded: {len(regulations)}")
print(f"Total documents in knowledge base: {len(all_documents)}")

Fraud patterns loaded: 7
Regulations loaded: 5
Total documents in knowledge base: 12


# **Embedded documents into ChromoDB**

In [32]:
from sentence_transformers import SentenceTransformer
import chromadb

#step1 loading the embedding model
embedder=SentenceTransformer("all-MiniLM-L6-v2")

#step2 create chromaDb client and connection
client=chromadb.Client()
collection=client.get_or_create_collection(name="finished_knowlege")

#step 3 embed all documents and store int in chromadb
for i,doc in enumerate(all_documents):
  embedding=embedder.encode(doc).tolist()
  collection.add(documents=[doc],embeddings=[embedding], ids=[f"doc_{i}"])
  print(f"  Stored doc_{i}: {doc[:60]}...")

print(f"\nTotal documents stored in ChromaDB: {collection.count()}")
print("Knowledge base ready ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Stored doc_0: High velocity transactions — more than 5 transactions in one...
  Stored doc_1: Transactions originating from high-risk countries such as Ni...
  Stored doc_2: Transactions occurring between midnight and 4am are statisti...
  Stored doc_3: Unusually high transaction amounts above $600 on cards with ...
  Stored doc_4: Device mismatch — transaction made from a device not previou...
  Stored doc_5: Transaction location more than 500km from the cardholder's h...
  Stored doc_6: The combination of high amount, foreign country, late night ...
  Stored doc_7: PCI-DSS Requirement 10: All transactions must be logged with...
  Stored doc_8: PCI-DSS Requirement 6: Transactions exceeding $500 from high...
  Stored doc_9: Basel III Framework: Financial institutions must maintain re...
  Stored doc_10: FFIEC Guidelines: Account takeover fraud requires immediate ...
  Stored doc_11: Regulation E (Electronic Fund Transfer Act): Banks must inve...

Total documents stored in ChromaDB: 1

# **Similarity Search**

In [33]:
query="""
Transaction TXN-0000004:Amount $900, country Romania, hour 2am, velocity 10 transactions in last hour, distance 4800km from home, device mismatch detected.
"""

query_embedding=embedder.encode(query).tolist()

results=collection.query(query_embeddings=[query_embedding],n_results=3)

print("Top 3 most relevant documents retrieved:\n")
for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i+1}:")
    print(f"  {doc[:120]}...")
    print()

Top 3 most relevant documents retrieved:

Result 1:
  Transaction location more than 500km from the cardholder's home address is a strong fraud signal, especially combined wi...

Result 2:
  The combination of high amount, foreign country, late night hour, and device mismatch creates a compound fraud signal. W...

Result 3:
  High velocity transactions — more than 5 transactions in one hour from the same card indicates card testing. Fraudsters ...



# **Connect GROQ LLM**

In [34]:
from google.colab import userdata
from groq import Groq

api_key=userdata.get("Groq_token")

groq_client=Groq(api_key=api_key)

response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Say 'FinShield AI connected' and nothing else."}
    ]
)

print(response.choices[0].message.content)


FinShield AI connected.


# **Full rag pipeline**

In [35]:
def investigate_transaction(transaction: dict) -> str:

    # Step 1: Convert transaction to text
    transaction_text = f"""
    Transaction ID: {transaction['id']}
    Amount: ${transaction['amount']}
    Country: {transaction['country']}
    Hour: {transaction['hour']}am
    Velocity: {transaction['velocity']} transactions in last hour
    Distance from home: {transaction['distance']}km
    Device mismatch: {transaction['device_mismatch']}
    """

    # Step 2: Retrieve relevant fraud patterns from ChromaDB
    query_embedding = embedder.encode(transaction_text).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )
    retrieved_patterns = "\n".join(results["documents"][0])

    # Step 3: Build prompt with transaction + retrieved context
    prompt = f"""
    You are a fraud investigation AI at a major bank.

    TRANSACTION TO INVESTIGATE:
    {transaction_text}

    INSTRUCTIONS:
    - Only flag as FRAUDULENT if multiple high-risk signals are present
    - $85 is a normal amount, $600+ is high risk
    - USA, UK are safe countries. Romania, Nigeria, Ukraine are high risk
    - Device mismatch False = safe signal
    - Distance under 50km = safe signal
    - Velocity under 3 = safe signal

    RELEVANT FRAUD PATTERNS FROM KNOWLEDGE BASE:
    {retrieved_patterns}

    Based on the transaction details and the fraud patterns above,
    provide a clear fraud investigation report. Include:
    1. Fraud verdict (FRAUDULENT / LEGITIMATE)
    2. Risk score (0-100)
    3. Key reasons why
    4. Recommended action
    """

    # Step 4: Send to Groq LLM
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


# Test with a suspicious transaction
suspicious_transaction = {
    "id": "TXN-0000004",
    "amount": 900,
    "country": "Romania",
    "hour": 2,
    "velocity": 10,
    "distance": 4800,
    "device_mismatch": True
}

print("=" * 60)
print("FINSHIELD AI — FRAUD INVESTIGATION REPORT")
print("=" * 60)
result = investigate_transaction(suspicious_transaction)
print(result)

FINSHIELD AI — FRAUD INVESTIGATION REPORT
**Fraud Investigation Report**

**Transaction ID:** TXN-0000004
**Investigation Date:** [Current Date]

**Fraud Verdict:** FRAUDULENT
**Risk Score:** 92%

**Key Reasons Why:**

1. **Large Transaction Amount:** The transaction amount of $900 is considered high-risk, as it exceeds $600.
2. **Foreign Country:** The transaction originated from Romania, a high-risk country.
3. **Late Night Hour:** The transaction occurred at 2am, which is considered a high-risk hour.
4. **Device Mismatch:** There is a device mismatch, which suggests a potential spoofing or phishing attempt.
5. **Significant Distance from Home Address:** The transaction distance of 4800km from the cardholder's home address is a strong fraud signal, suggesting card cloning or data theft.

**Additional Context:** The high transaction velocity (10 transactions in the last hour) and the fact that multiple high-risk signals are present, compound the fraud probability exceeds 95%.

**Recom

In [36]:
# ============================================================
# CELL 8 — Test with Legitimate Transaction
# ============================================================

legitimate_transaction = {
    "id": "TXN-0000099",
    "amount": 85,
    "country": "USA",
    "hour": 3,  # 3pm
    "velocity": 1,
    "distance": 12,
    "device_mismatch": False
}

print("=" * 60)
print("FINSHIELD AI — FRAUD INVESTIGATION REPORT")
print("=" * 60)
result = investigate_transaction(legitimate_transaction)
print(result)

FINSHIELD AI — FRAUD INVESTIGATION REPORT
**Fraud Investigation Report: Transaction ID TXN-0000099**

**Fraud Verdict:** LEGITIMATE

**Risk Score:** 20

**Key Reasons Why:**

1. The transaction amount of $85 is below the high-risk threshold of $600.
2. The transaction was made in the USA, a safe country.
3. The hour of the transaction, 3am, is not a strong indicator of fraud by itself. Although it is late night, it does not fall into a high-risk hour category.
4. The transaction velocity of 1 transaction in the last hour does not indicate a high-risk activity.
5. There is no device mismatch, indicating the transaction was made from a safe device.
6. The distance from home of 12km is below the safe threshold of 50km.
7. Although the country of origin and device mismatch combination can create a compound fraud signal, this transaction does not fit the exact pattern as described in the knowledge base.

**Recommended Action:** No further action is required on this transaction. It can be pr

# **LangGraph Agent Setup**

In [37]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

# Define the State — this travels through every node
class FraudInvestigationState(TypedDict):
    transaction: dict          # raw transaction data
    transaction_text: str      # formatted text version
    risk_score: int            # 0-100 risk score
    retrieved_patterns: List[str]  # from ChromaDB
    verdict: str               # FRAUDULENT / LEGITIMATE
    report: str                # final investigation report

print("LangGraph imported ✅")
print("State schema defined ✅")
print()
print("State fields:")
for field in FraudInvestigationState.__annotations__:
    print(f"  - {field}")

LangGraph imported ✅
State schema defined ✅

State fields:
  - transaction
  - transaction_text
  - risk_score
  - retrieved_patterns
  - verdict
  - report


# **Define the 3 Agent Nodes**

In [38]:
# NODE 1: Analyze Risk Score
def analyze_risk(state: FraudInvestigationState) -> FraudInvestigationState:
    print("🔍 Node 1: Analyzing risk score...")
    t = state["transaction"]

    score = 0
    if t["amount"] > 600:        score += 25
    if t["country"] in ["Romania", "Nigeria", "Ukraine"]: score += 25
    if t["hour"] <= 4:           score += 20
    if t["velocity"] >= 5:       score += 15
    if t["distance"] > 500:      score += 10
    if t["device_mismatch"]:     score += 15

    score = min(score, 100)  # cap at 100

    transaction_text = f"""
    Transaction ID: {t['id']}
    Amount: ${t['amount']}
    Country: {t['country']}
    Hour: {t['hour']}am
    Velocity: {t['velocity']} transactions/hour
    Distance from home: {t['distance']}km
    Device mismatch: {t['device_mismatch']}
    """

    print(f"   Risk score calculated: {score}/100")
    return {**state, "risk_score": score, "transaction_text": transaction_text}


# NODE 2: Retrieve Patterns from ChromaDB
def retrieve_patterns(state: FraudInvestigationState) -> FraudInvestigationState:
    print("📚 Node 2: Retrieving fraud patterns from ChromaDB...")

    query_embedding = embedder.encode(state["transaction_text"]).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )
    patterns = results["documents"][0]

    print(f"   Retrieved {len(patterns)} relevant patterns")
    return {**state, "retrieved_patterns": patterns}


# NODE 3: Generate Final Report
def generate_report(state: FraudInvestigationState) -> FraudInvestigationState:
    print("📝 Node 3: Generating investigation report with Groq...")

    patterns_text = "\n".join(state["retrieved_patterns"])
    verdict_hint = "FRAUDULENT" if state["risk_score"] >= 50 else "LEGITIMATE"

    prompt = f"""
    You are a fraud investigation AI at a major bank.

    TRANSACTION:
    {state["transaction_text"]}

    RISK SCORE: {state["risk_score"]}/100

    RELEVANT FRAUD PATTERNS:
    {patterns_text}

    INSTRUCTIONS:
    - The risk score is already calculated: {state["risk_score"]}/100
    - Risk score 0-49 = LEGITIMATE, no exceptions
    - Risk score 50-100 = FRAUDULENT, no exceptions
    - Your verdict MUST match: {verdict_hint}
    - Do not override the verdict under any circumstances
    - Just explain why the score is what it is

    Provide:
    1. Fraud verdict ({verdict_hint})
    2. Risk score confirmation
    3. Key reasons (bullet points)
    4. Recommended action
    """

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    report = response.choices[0].message.content
    verdict = "FRAUDULENT" if state["risk_score"] >= 50 else "LEGITIMATE"

    print(f"   Verdict: {verdict}")
    return {**state, "report": report, "verdict": verdict}


print("All 3 nodes defined ✅")
print("  - analyze_risk")
print("  - retrieve_patterns")
print("  - generate_report")

All 3 nodes defined ✅
  - analyze_risk
  - retrieve_patterns
  - generate_report


# **Build and Compile the LangGraph Agent**

In [39]:
# Step 1: Create the graph
graph = StateGraph(FraudInvestigationState)

# Step 2: Add nodes
graph.add_node("analyze_risk", analyze_risk)
graph.add_node("retrieve_patterns", retrieve_patterns)
graph.add_node("generate_report", generate_report)

# Step 3: Add edges (the flow)
graph.set_entry_point("analyze_risk")
graph.add_edge("analyze_risk", "retrieve_patterns")
graph.add_edge("retrieve_patterns", "generate_report")
graph.add_edge("generate_report", END)

# Step 4: Compile
agent = graph.compile()

print("LangGraph agent compiled ✅")
print()
print("Agent flow:")
print("  START")
print("    ↓")
print("  analyze_risk")
print("    ↓")
print("  retrieve_patterns")
print("    ↓")
print("  generate_report")
print("    ↓")
print("  END")

LangGraph agent compiled ✅

Agent flow:
  START
    ↓
  analyze_risk
    ↓
  retrieve_patterns
    ↓
  generate_report
    ↓
  END


# **Run the LangGraph Agent**

In [40]:

# Test 1: Suspicious transaction
print("=" * 60)
print("TEST 1 — SUSPICIOUS TRANSACTION")
print("=" * 60)

suspicious = {
    "id": "TXN-0000004",
    "amount": 900,
    "country": "Romania",
    "hour": 2,
    "velocity": 10,
    "distance": 4800,
    "device_mismatch": True
}

initial_state = {
    "transaction": suspicious,
    "transaction_text": "",
    "risk_score": 0,
    "retrieved_patterns": [],
    "verdict": "",
    "report": ""
}

result1 = agent.invoke(initial_state)
print()
print(f"FINAL VERDICT: {result1['verdict']}")
print(f"RISK SCORE: {result1['risk_score']}/100")
print()
print("FULL REPORT:")
print(result1["report"])

# Test 2: Legitimate transaction
print()
print("=" * 60)
print("TEST 2 — LEGITIMATE TRANSACTION")
print("=" * 60)

legitimate = {
    "id": "TXN-0000099",
    "amount": 85,
    "country": "USA",
    "hour": 3,
    "velocity": 1,
    "distance": 12,
    "device_mismatch": False
}

initial_state2 = {
    "transaction": legitimate,
    "transaction_text": "",
    "risk_score": 0,
    "retrieved_patterns": [],
    "verdict": "",
    "report": ""
}

result2 = agent.invoke(initial_state2)
print()
print(f"FINAL VERDICT: {result2['verdict']}")
print(f"RISK SCORE: {result2['risk_score']}/100")
print()
print("FULL REPORT:")
print(result2["report"])

TEST 1 — SUSPICIOUS TRANSACTION
🔍 Node 1: Analyzing risk score...
   Risk score calculated: 100/100
📚 Node 2: Retrieving fraud patterns from ChromaDB...
   Retrieved 3 relevant patterns
📝 Node 3: Generating investigation report with Groq...
   Verdict: FRAUDULENT

FINAL VERDICT: FRAUDULENT
RISK SCORE: 100/100

FULL REPORT:
**Fraud Verdict:** FRAUDULENT

**Risk Score Confirmation:** 100/100

**Key Reasons:**

* Transaction location is more than 500km from the cardholder's home address, indicating card cloning or data theft.
* The combination of a high amount ($900), foreign country (Romania), late night hour (2am), and device mismatch creates a compound fraud signal.
* The transaction originated from a high-risk country (Romania) for a cardholder based in a low-risk country (likely outside of Romania and possibly the USA or UK).

**Recommended Action:** IMMEDIATELY FREEZE THE ACCOUNT AND REPORT THE TRANSACTION TO LAW ENFORCEMENT, AS WELL AS TO THE CARDHOLDER'S BANK FOR FURTHER INVESTIGA